In [1]:
import os
import re
import pandas as pd
from IPython.display import display

def read_csv_auto(path):
    for enc in ["utf-8", "utf-8-sig", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin-1")

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    return text

INTENT_CONFIG = {
    "decision": {
        "tower.decision.right": [["right"], ["2"], ["two"], ["second"]],
        "tower.decision.left": [["left"], ["1"], ["number", "one"], ["first"]],
        "tower.decision.same": [["both"], ["together"], ["same", "time"], ["one", "and", "two"]]
    },
    "reason": {
        "tower.reason.tall": [["tall"], ["taller"], ["big"], ["bigger"], ["high"], ["higher"]],
        "tower.reason.weight": [["weight"], ["weighs"], ["top"], ["heavy"], ["heavier"]],
        "tower.reason.thinnerbase": [["thin"], ["thinner"], ["small"], ["smaller"], ["base"], ["bottom"]],
        "tower.reason.notsymmetrical": [["symmetry"], ["symmetric"], ["symmetrical"], ["both", "sides"], ["not", "the", "same"], ["asymmetrical"], ["asymmetric"], ["not", "same"]]
    }
}

def determine_intent(transcript, state):
    if not transcript or str(transcript).strip() == "":
        return "default.noaudio"

    clean = normalize_text(transcript)
    words = clean.split()

    state_intents = INTENT_CONFIG.get(state, {})

    for intent_name, groups in state_intents.items():
        for group in groups:
            if all(any(re.fullmatch(re.escape(k), w) for w in words) for k in group):
                return intent_name

    return "default.unknown"

print("Helpers and intent config ready.")

Helpers and intent config ready.


In [2]:
INPUT_CSV = "./reason_prediction_audio_log.csv"

df = read_csv_auto(INPUT_CSV)

transcript_col = "Manual Transcription"
decision_col = "manual intent [decision]"
reason_col = "manual intent [reason]"

if transcript_col not in df.columns:
    raise ValueError(f"Column '{transcript_col}' not found in CSV. Available columns: {list(df.columns)}")

insert_pos = df.columns.get_loc(transcript_col) + 1

for idx, row in df.iterrows():
    transcript = str(row.get(transcript_col, '')).strip()
    if transcript in ["", "nan", "NO SPOKEN WORD"]:
        df.at[idx, decision_col] = "default.noaudio"
        df.at[idx, reason_col] = "default.noaudio"
    else:
        df.at[idx, decision_col] = determine_intent(transcript, "decision")
        df.at[idx, reason_col] = determine_intent(transcript, "reason")

cols = list(df.columns)
for col_name in [decision_col, reason_col]:
    if col_name in cols:
        cols.remove(col_name)

cols.insert(insert_pos, decision_col)
cols.insert(insert_pos + 1, reason_col)
df = df[cols]

classified = df[df[transcript_col].notna() & (df[transcript_col] != "")].shape[0]
print(f"Classified {classified} rows using '{transcript_col}' column.")
print(f"Inserted '{decision_col}' and '{reason_col}' right after '{transcript_col}'.")
display(df[[transcript_col, decision_col, reason_col]].head(20))

Classified 124 rows using 'Manual Transcription' column.
Inserted 'manual intent [decision]' and 'manual intent [reason]' right after 'Manual Transcription'.


,Manual Transcription,manual intent [decision],manual intent [reason]
0,Because it has a thinner base,default.unknown,tower.reason.thinnerbase
1,Because it had a thinner base,default.unknown,tower.reason.thinnerbase
2,NO SPOKEN WORD,default.noaudio,default.noaudio
3,Because it has more weight on top than bottom,default.unknown,tower.reason.weight
4,Because it has a thinner base,default.unknown,tower.reason.thinnerbase
5,Yeah. In that case we have to make it cause i...,default.unknown,tower.reason.tall
6,Because that tower is taller than the other tower,default.unknown,tower.reason.tall
7,Because it has a smaller base,default.unknown,tower.reason.thinnerbase
8,Um It's not the same on both sides,tower.decision.same,tower.reason.notsymmetrical
9,no the bottom is,default.unknown,tower.reason.thinnerbase


In [3]:
df.to_csv(INPUT_CSV, index=False)
print(f"Saved results to {INPUT_CSV}")

Saved results to ./reason_prediction_audio_log.csv
